# OpenRAN Load Balancing & Version Evaluation Suite
### Individual 2-Axis Evaluation Plots
- **OAI (OpenAirInterface)** Load Balancing & Scalability Metrics
- **srsRAN (v25.10 / v23.11 / v22.10)** Load Balancing Metrics
- **Comparative 2-Axis Plots**: MCS, System Load, RAPL Power, RAM Usage, Throughput, and Latency

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set clean publication-ready plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 15,
    'lines.linewidth': 2.2,
    'grid.alpha': 0.5
})

# GitHub raw stream fallback (allows running in Colab without manual git clone)
GITHUB_RAW = "https://raw.githubusercontent.com/saishresphd/openran-version/working_v25/"

def load_csv_dataset(rel_path):
    local_candidates = [
        rel_path,
        os.path.basename(rel_path),
        os.path.join("results/ver_eval", rel_path),
        os.path.join("results/ver_eval", os.path.basename(rel_path)),
        os.path.expanduser(f"~/Desktop/{os.path.basename(rel_path)}")
    ]
    for p in local_candidates:
        if os.path.exists(p):
            try:
                df = pd.read_csv(p)
                if not df.empty and 'ue_id' in df.columns:
                    print(f"✓ Loaded local: {p} ({len(df)} rows)")
                    return df
            except Exception:
                pass
    # Fallback to direct raw GitHub stream
    url = GITHUB_RAW + rel_path
    try:
        df = pd.read_csv(url)
        print(f"✓ Loaded remote GitHub: {url} ({len(df)} rows)")
        return df
    except Exception as e:
        print(f"⚠️ Remote fetch failed for {url}: {e}")
        return pd.DataFrame()

df_oai = load_csv_dataset('results/ver_eval/oai/ue_results_50.csv')
df_srs_v25 = load_csv_dataset('results/ver_eval/v25_10_ue_results_50.csv')
df_srs_v23 = load_csv_dataset('results/ver_eval/v23_11/ue_results_50.csv')
df_srs_v22 = load_csv_dataset('results/ver_eval/v22_10/ue_results_50.csv')
df_lb = load_csv_dataset('results/ver_eval/openran_lb_publication_dataset_50ue.csv')

print(f"\nSummary -> OAI: {len(df_oai)} rows, srsRAN v25: {len(df_srs_v25)} rows, LB Data: {len(df_lb)} rows")

## 1. OAI Individual 2-Axis Evaluation Plots

In [ ]:
# Plot 1.1: OAI - Throughput vs Active UE Load
plt.figure(figsize=(8, 5))
x = df_oai['ue_id']
y = df_oai['dl_10m_mbps']
plt.plot(x, y, color='#0f62fe', marker='o', label='OAI DL Throughput (10M target)')
plt.axvline(x=15, color='red', linestyle='--', label='Cell Saturation Boundary (15 UEs)')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Per-UE DL Throughput (Mbps)')
plt.title('OAI: Per-UE DL Throughput vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.2: OAI - RAPL Server Power vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_oai['ue_id'], df_oai['gnb_rapl_total_w'], color='#da1e28', marker='s', label='gNB Total RAPL Power')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('RAPL Package Power (Watts)')
plt.title('OAI: Server RAPL Power Consumption vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.3: OAI - Downlink & Uplink MCS vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_oai['ue_id'], df_oai['ran_dl_mcs'], color='#198038', marker='^', label='DL MCS Index')
plt.plot(df_oai['ue_id'], df_oai['ran_ul_mcs'], color='#8a3800', marker='v', linestyle='--', label='UL MCS Index')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Modulation & Coding Scheme (MCS Index)')
plt.title('OAI: Modulation & Coding Scheme (MCS) vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.4: OAI - RAM Memory Footprint vs Active UE Load
plt.figure(figsize=(8, 5))
ram_mb = df_oai['ran_proc_rmem_kb'] / 1024.0
plt.plot(df_oai['ue_id'], ram_mb, color='#8a3ffc', marker='D', label='OAI Process Resident RAM')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Resident Memory (MB)')
plt.title('OAI: gNB Resident RAM Footprint vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1.5: OAI - System Load vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_oai['ue_id'], df_oai['gnb_load1'], color='#00539a', marker='o', label='CPU Load 1-min Avg')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Linux System Load (Load1)')
plt.title('OAI: Host Linux System Load vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

## 2. srsRAN Individual 2-Axis Evaluation Plots (v25.10)

In [ ]:
# Plot 2.1: srsRAN v25.10 - Throughput vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['dl_10m_mbps'], color='#0f62fe', marker='o', label='srsRAN v25 DL Tput')
plt.axvline(x=14, color='red', linestyle='--', label='PRB Saturation Threshold (14 UEs)')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Per-UE DL Throughput (Mbps)')
plt.title('srsRAN v25.10: Per-UE DL Throughput vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.2: srsRAN v25.10 - Server RAPL Power vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['gnb_rapl_total_w'], color='#da1e28', marker='s', label='gNB Total RAPL Power')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('RAPL Package Power (Watts)')
plt.title('srsRAN v25.10: Server RAPL Power Consumption vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.3: srsRAN v25.10 - DL & UL MCS vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['ran_dl_mcs'], color='#198038', marker='^', label='DL MCS')
plt.plot(df_srs_v25['ue_id'], df_srs_v25['ran_ul_mcs'], color='#8a3800', marker='v', linestyle='--', label='UL MCS')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Modulation & Coding Scheme (MCS Index)')
plt.title('srsRAN v25.10: MCS Adaptation vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.4: srsRAN v25.10 - Resident RAM Footprint vs Active UE Load
plt.figure(figsize=(8, 5))
ram_srs_mb = df_srs_v25['ran_proc_rmem_kb'] / 1024.0
plt.plot(df_srs_v25['ue_id'], ram_srs_mb, color='#8a3ffc', marker='D', label='srsRAN Resident RAM')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Resident Memory (MB)')
plt.title('srsRAN v25.10: Process RAM Footprint vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2.5: srsRAN v25.10 - CPU Utilization Percentage vs Active UE Load
plt.figure(figsize=(8, 5))
plt.plot(df_srs_v25['ue_id'], df_srs_v25['gnb_srsenb_cpu_pct'], color='#ff832b', marker='p', label='srsgNB Process CPU %')
plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Process CPU Utilization (%)')
plt.title('srsRAN v25.10: Process CPU Utilization vs Active UE Load')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Comparison Plots (OAI vs srsRAN v25.10 vs v23.11 vs v22.10)

In [ ]:
# Plot 3.1: Comparative - Per-UE DL Throughput vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['dl_10m_mbps'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['dl_10m_mbps'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['dl_10m_mbps'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['dl_10m_mbps'][:30], color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Per-UE DL Throughput (Mbps)')
plt.title('Cross-Stack Comparison: Per-UE DL Throughput vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.2: Comparative - Server RAPL Power vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['gnb_rapl_total_w'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['gnb_rapl_total_w'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['gnb_rapl_total_w'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['gnb_rapl_total_w'][:30], color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('RAPL Package Power (Watts)')
plt.title('Cross-Stack Comparison: Server RAPL Power vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.3: Comparative - Downlink MCS vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['ran_dl_mcs'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10 DL MCS')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['ran_dl_mcs'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11 DL MCS')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['ran_dl_mcs'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10 DL MCS')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['ran_dl_mcs'][:30], color='#fa4d56', marker='s', label='OAI DL MCS')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Modulation & Coding Scheme (MCS Index)')
plt.title('Cross-Stack Comparison: DL MCS vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.4: Comparative - Resident RAM Footprint vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['ran_proc_rmem_kb'][:30]/1024.0, color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['ran_proc_rmem_kb'][:30]/1024.0, color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['ran_proc_rmem_kb'][:30]/1024.0, color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['ran_proc_rmem_kb'][:30]/1024.0, color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Resident Memory (MB)')
plt.title('Cross-Stack Comparison: Process Resident RAM vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.5: Comparative - Host System Load1 vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['gnb_load1'][:30], color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['gnb_load1'][:30], color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['gnb_load1'][:30], color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['gnb_load1'][:30], color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Linux System Load (Load1)')
plt.title('Cross-Stack Comparison: Host System Load vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3.6: Comparative - Attach Latency vs Load
plt.figure(figsize=(9, 5.5))
if not df_srs_v25.empty:
    plt.plot(df_srs_v25['ue_id'][:30], df_srs_v25['attach_ms'][:30]/1000.0, color='#0f62fe', marker='o', label='srsRAN v25.10')
if not df_srs_v23.empty:
    plt.plot(df_srs_v23['ue_id'][:30], df_srs_v23['attach_ms'][:30]/1000.0, color='#1192e8', marker='^', linestyle='--', label='srsRAN v23.11')
if not df_srs_v22.empty:
    plt.plot(df_srs_v22['ue_id'][:30], df_srs_v22['attach_ms'][:30]/1000.0, color='#005d5d', marker='x', linestyle=':', label='srsRAN v22.10')
if not df_oai.empty:
    plt.plot(df_oai['ue_id'][:30], df_oai['attach_ms'][:30]/1000.0, color='#fa4d56', marker='s', label='OpenAirInterface (OAI)')

plt.xlabel('Active Connected UEs (Load)')
plt.ylabel('Control Plane Attach Duration (s)')
plt.title('Cross-Stack Comparison: UE Attach Latency vs UE Load')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Load Balancing Strategy Comparisons (Unmanaged vs Standard 3GPP vs BPEA-LB)

In [ ]:
# Plot 4.1: Load Balancing Strategies - Per-UE DL Throughput Post-LB
plt.figure(figsize=(9, 5.5))
if not df_lb.empty:
    for strategy, col, marker in [
        ('Unmanaged_Single_gNB', '#da1e28', 'x'),
        ('Standard_3GPP_Early_Handover', '#f1c21b', '^'),
        ('BPEA_LB_Gated_Handover', '#198038', 'o')
    ]:
        sub = df_lb[df_lb['lb_strategy'] == strategy]
        if not sub.empty:
            plt.plot(sub['ue_id'], sub['post_lb_tput_mbps'], label=strategy.replace('_', ' '), color=col, marker=marker)

plt.axhline(y=10.0, color='blue', linestyle=':', label='SLA Target (10 Mbps)')
plt.xlabel('UE ID (Sequential Cell Load)')
plt.ylabel('Post-LB DL Throughput (Mbps)')
plt.title('Load Balancing Comparison: Post-Migration Per-UE Throughput')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4.2: Load Balancing Strategies - Total Power (gNB1 + gNB2)
plt.figure(figsize=(9, 5.5))
if not df_lb.empty:
    for strategy, col, marker in [
        ('Unmanaged_Single_gNB', '#da1e28', 'x'),
        ('Standard_3GPP_Early_Handover', '#f1c21b', '^'),
        ('BPEA_LB_Gated_Handover', '#198038', 'o')
    ]:
        sub = df_lb[df_lb['lb_strategy'] == strategy]
        if not sub.empty and 'total_system_power_w' in sub.columns:
            plt.plot(sub['ue_id'], sub['total_system_power_w'], label=strategy.replace('_', ' '), color=col, marker=marker)

plt.xlabel('UE ID (Sequential Cell Load)')
plt.ylabel('Total System Power (gNB1 + gNB2 Watts)')
plt.title('Load Balancing Comparison: Total Cluster Power Consumption')
plt.legend()
plt.tight_layout()
plt.show()